In [ ]:
from autocleanse.utils import load_csv
from autocleanse.cleaner import handle_missing_data
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LinearRegression
import pandas as pd

The history saving thread hit an unexpected error (DatabaseError('database disk image is malformed')).History will not be written to the database.


In [2]:
df = load_csv("../data/train.csv")
df = handle_missing_data(df)
df['Is_Alone'] = ((df['SibSp'] + df['Parch']) == 0).astype(int)
df = df.drop(columns=['Name', 'Ticket', 'Parch', 'SibSp'], axis=1)
columns = [column for column in df.columns if df[column].nunique() / len(df) < 0.05]
columns.remove('Survived')
print(columns)

Started Cleaning the Data...
Finished Cleaning the Data!
['Pclass', 'Sex', 'Embarked', 'Is_Alone']


In [3]:
df_encoded = pd.get_dummies(df)

In [4]:
X = df_encoded.drop('Survived', axis= 1)
y = df_encoded['Survived']

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y)

In [6]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
model = LinearRegression()
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)

In [8]:
# Look at raw predictions
print(y_pred[:10])  # First 10 predicted labels (0 or 1)
print(y_test[:10].values)  # First 10 actual labels

[0 0 1 1 1 1 1 1 0 0]
[0 0 0 1 0 1 1 1 0 0]


In [9]:
acc = accuracy_score(y_test, y_pred=y_pred)
print(f"Accuracy: {acc:.2f}")

Accuracy: 0.80


In [10]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)
print(cm)


[[126  10]
 [ 34  53]]


In [11]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

           0       0.79      0.93      0.85       136
           1       0.84      0.61      0.71        87

    accuracy                           0.80       223
   macro avg       0.81      0.77      0.78       223
weighted avg       0.81      0.80      0.79       223



In [12]:
import joblib

joblib.dump(model, '../data/titanic_model.pkl')
joblib.dump(scaler, '../data/titanic_scaler.pkl')
# After creating X_train and before scaling
training_columns = X_train.columns
joblib.dump(training_columns, '../data/training_columns.pkl')


['../data/training_columns.pkl']